# Lesson 01 - AI 代理人介紹

歡迎來到<strong>初學者的 AI 代理人</strong>課程的第一課！

<strong>AI 代理人</strong>是一個使用大型語言模型（LLM）作為其推理引擎的程式，能夠在現實世界中採取<em>行動</em> —— 呼叫 API、查詢資料庫或執行程式碼 —— 以代表使用者完成目標。

在本筆記本中，您將建立您的第一個代理人：一個推薦旅遊目的地的<strong>旅遊代理人</strong>。在過程中，您將學習如何：

1. 使用<strong>Microsoft 代理人框架</strong>連接到 Azure AI Foundry 代理人服務。
2. 給代理人一個<strong>工具</strong> —— 一個它可以呼叫的純 Python 函數。
3. 執行代理人並檢查其回應。
4. 逐字串串流代理人的回應。


## Setup

在執行此筆記本之前，請確保您已：

1. **擁有 Azure AI Foundry 專案** 且已部署聊天模型（例如 `gpt-4o-mini`）。
2. **已使用 Azure CLI 登入** — 在終端機執行 `az login`。
3. **設定所需的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 專案端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您部署的模型名稱。

下面的儲存格會安裝您需要的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 建立您的第一個代理人

代理人需要兩樣東西：

- <strong>指示</strong>，告訴它<em>是誰</em>以及<em>如何行事</em>（系統提示）。
- <strong>工具</strong> — 使用 `@tool` 裝飾的 Python 函數，代理人可以調用這些函數來取得資訊或執行操作。

以下我們定義了一個簡單的工具，會回傳熱門度假目的地清單。當使用者詢問旅遊建議時，代理人會使用此工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

為了提供更具互動性的體驗，您可以<strong>串流</strong>代理的回應。代理不需等待完整回覆即會開始輸出生成的文字片段。這在聊天介面中特別有用，您可以即時顯示輸出內容。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

在本課程中，您學會了如何：

- <strong>建立一個提供者</strong>，透過 `AzureAIProjectAgentProvider` 連接到 Azure AI Foundry Agent 服務。
- **使用 `@tool` 裝飾器定義工具**，使代理能呼叫您的 Python 函式。
- <strong>運行代理</strong>，傳入使用者訊息並列印其回應。
- <strong>串流回應</strong>，以獲得即時輸出。

在下一課中，我們將更深入探討代理框架，並學習如何賦予代理更強大的工具和多步驟推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
